In [ ]:
from abc import ABC, abstractmethod


class User(ABC):
    """Common identity. Phone is private, nobody edits it directly."""

    def __init__(self, uid, name, phone, email, city):
        self.uid = uid
        self.name = name
        self.email = email
        self.city = city
        self.__phone = phone

    @property
    def phone(self):
        return self.__phone

    def update_phone(self, new_phone):
        if not (str(new_phone).isdigit() and len(str(new_phone)) == 10):
            raise ValueError("phone must be 10 digits")
        self.__phone = new_phone

    @abstractmethod
    def role(self):
        """Short label: Customer / Rider."""

    def __str__(self):
        return f"{self.role():<9} {self.name:<15} {self.city}"

class Customer(User):
    def __init__(self, cust_id, name, phone, email, city, dob, gender, pincode):
        super().__init__(cust_id, name, phone, email, city)
        self.dob = dob
        self.gender = gender
        self.pincode = pincode
        self.cart = Cart(self)
        self.orders = []

    def role(self):
        return "Customer"

    def place_order(self, inventory):
        order = self.cart.checkout(inventory)
        if order:
            self.orders.append(order)
        return order


class Rider(User):
    def __init__(self, rider_id, name, phone, email, city):
        super().__init__(rider_id, name, phone, email, city)
        self.is_free = True
        self.completed = 0

    def role(self):
        return "Rider"

    def accept(self):
        if not self.is_free:
            return False
        self.is_free = False
        return True

    def finish(self):
        self.is_free = True
        self.completed += 1

from abc import ABC, abstractmethod


class Product(ABC):
    def __init__(self, product_id, prod_name, price, category):
        self.product_id = product_id
        self.prod_name = prod_name
        self.price = price
        self.category = category

    @abstractmethod
    def max_per_order(self):
        """Each type decides how many one order may carry."""

    @abstractmethod
    def handling(self):
        """Short note for the rider: cold bag, fragile, normal."""

    def __str__(self):
        return f"{self.prod_name:<26} Rs {self.price:>7.2f}  [{self.category}]"


class PerishableProduct(Product):
    """Dairy, Vegetables, Fruits, Bakery: short life, small limit."""

    def max_per_order(self):
        return 5

    def handling(self):
        return "Keep cold / fresh"


class PackagedProduct(Product):
    """Snacks, Beverages, Staples, Household: long life, bigger limit."""

    def max_per_order(self):
        return 10

    def handling(self):
        return "Normal"


PERISHABLE = {"Dairy", "Vegetables", "Fruits", "Bakery"}


def make_product(product_id, prod_name, price, category):
    cls = PerishableProduct if category in PERISHABLE else PackagedProduct
    return cls(product_id, prod_name, price, category)

class Inventory:
    def __init__(self):
        self.__stock = {}      # product_id -> qty (private)
        self._log = []         # (kind, product_id, qty, left)

    def stock_of(self, product_id):
        return self.__stock.get(product_id, 0)

    def _apply(self, product_id, qty, kind):
        """The ONE place stock moves."""
        self.__stock[product_id] = self.stock_of(product_id) + qty
        self._log.append((kind, product_id, qty, self.__stock[product_id]))

    def restock(self, product_id, qty):
        if qty <= 0:
            raise ValueError("restock must be positive")
        self._apply(product_id, qty, "restock")

    def can_sell(self, product_id, qty):
        return 0 < qty <= self.stock_of(product_id)

    def sell(self, product_id, qty):
        if not self.can_sell(product_id, qty):
            return False
        self._apply(product_id, -qty, "sell")
        return True

    def statement(self):
        return list(self._log)

class Cart:
    def __init__(self, customer):
        self.customer = customer
        self.items = {}   # Product -> qty

    def add(self, product, qty=1):
        have = self.items.get(product, 0)
        if have + qty > product.max_per_order():
            return f"Max {product.max_per_order()} of {product.prod_name} per order."
        self.items[product] = have + qty
        return f"Added {qty} x {product.prod_name}."

    def remove(self, product):
        self.items.pop(product, None)

    def total(self):
        return sum(p.price * q for p, q in self.items.items())

    def checkout(self, inventory):
        if not self.items:
            return None
        for p, q in self.items.items():
            if not inventory.can_sell(p.product_id, q):
                return None
        for p, q in self.items.items():
            inventory.sell(p.product_id, q)
        order = Order(self.customer, dict(self.items))
        self.items.clear()
        return order


class Order:
    _next_id = 1

    def __init__(self, customer, items):
        self.order_id = Order._next_id
        Order._next_id += 1
        self.customer = customer
        self.items = items
        self.delivery = None

    def total(self):
        return sum(p.price * q for p, q in self.items.items())

    def __str__(self):
        return f"Order #{self.order_id} {self.customer.name:<14} Rs {self.total():,.2f}"


class Delivery:
    flow = ["pending", "out for delivery", "delivered"]
    _next_id = 1

    def __init__(self, order, rider):
        self.delivery_id = Delivery._next_id
        Delivery._next_id += 1
        self.order = order
        self.rider = rider
        self.__status = "pending"
        self.payment_status = "pending"
        order.delivery = self

    @property
    def status(self):
        return self.__status

    def advance(self):
        """The one place status changes."""
        i = self.flow.index(self.__status)
        if i == len(self.flow) - 1:
            return "Already delivered."
        nxt = self.flow[i + 1]
        if nxt == "out for delivery" and not self.rider.accept():
            return f"Rider {self.rider.name} is busy."
        if nxt == "delivered":
            self.rider.finish()
        self.__status = nxt
        return f"Delivery #{self.delivery_id}: {self.__status}"

    def mark_paid(self):
        self.payment_status = "paid"

    def __str__(self):
        return (f"Delivery #{self.delivery_id} order {self.order.order_id}  "
                f"rider {self.rider.name}  {self.status} / {self.payment_status}")


class Address:
    def __init__(self, address_id, label, line1, city, pincode):
        self.address_id = address_id
        self.label = label
        self.line1 = line1
        self.city = city
        self.pincode = pincode

    def is_serviceable(self, service_pincodes):
        return self.pincode in service_pincodes

    def __str__(self):
        return f"{self.label}: {self.line1}, {self.city} - {self.pincode}"


class Payment:
    methods = ["upi", "card", "cash"]
    _next_id = 1

    def __init__(self, order, method):
        if method not in self.methods:
            raise ValueError(f"method must be one of {self.methods}")
        self.payment_id = Payment._next_id
        Payment._next_id += 1
        self.order = order
        self.method = method
        self.amount = order.total()
        self.__status = "pending"

    @property
    def status(self):
        return self.__status

    def pay(self):
        if self.__status == "paid":
            return "Already paid."
        self.__status = "paid"
        return f"Payment #{self.payment_id}: paid Rs {self.amount:,.2f} by {self.method}"

    def refund(self):
        if self.__status != "paid":
            return "Only a paid payment can be refunded."
        self.__status = "refunded"
        return f"Payment #{self.payment_id}: refunded"

    def __str__(self):
        return f"Payment #{self.payment_id} {self.method:<5} Rs {self.amount:>8.2f} {self.status}"


class Coupon:
    def __init__(self, code, discount_percent, max_discount, min_order=0, active=True):
        self.code = code
        self.discount_percent = discount_percent
        self.max_discount = max_discount
        self.min_order = min_order
        self.active = active

    def is_valid(self, order_total):
        return self.active and order_total >= self.min_order

    def apply(self, order_total):
        if not self.is_valid(order_total):
            return order_total
        discount = order_total * self.discount_percent / 100
        discount = min(discount, self.max_discount)
        return round(order_total - discount, 2)

    def deactivate(self):
        self.active = False

    def __str__(self):
        return f"{self.code}: {self.discount_percent}% off, up to Rs {self.max_discount}"


class Review:
    _next_id = 1

    def __init__(self, order, rating, comment=""):
        self._check(rating)
        self.review_id = Review._next_id
        Review._next_id += 1
        self.order = order
        self.comment = comment
        self.__rating = rating

    def _check(self, value):
        if not (1 <= value <= 5):
            raise ValueError("rating must be between 1 and 5")

    @property
    def rating(self):
        return self.__rating

    def update_rating(self, new_rating):
        self._check(new_rating)
        self.__rating = new_rating

    def is_positive(self):
        return self.__rating >= 4

    def __str__(self):
        return (f"Review #{self.review_id} order {self.order.order_id}  "
                f"{self.__rating}/5  {self.comment}")


class Category:
    def __init__(self, name):
        self.name = name
        self.products = []

    def add_product(self, product):
        if product.category != self.name:
            raise ValueError(f"{product.prod_name} belongs to {product.category}, not {self.name}")
        self.products.append(product)

    def list_products(self):
        return list(self.products)

    def __str__(self):
        return f"{self.name}: {len(self.products)} products"


class Wishlist:
    def __init__(self, customer):
        self.customer = customer
        self.products = []

    def add(self, product):
        if product not in self.products:
            self.products.append(product)

    def remove(self, product):
        if product in self.products:
            self.products.remove(product)

    def move_to_cart(self, product, qty=1):
        if product not in self.products:
            return f"{product.prod_name} is not in the wishlist."
        message = self.customer.cart.add(product, qty)
        if message.startswith("Added"):
            self.products.remove(product)
        return message

    def __str__(self):
        return f"{self.customer.name}: {len(self.products)} products in wishlist"


class Notification:
    types = ["order", "delivery", "payment", "offer"]
    _next_id = 1

    def __init__(self, customer, message, kind="order"):
        self.notification_id = Notification._next_id
        Notification._next_id += 1
        self.customer = customer
        self.message = message
        self.kind = kind
        self.__is_read = False

    @property
    def is_read(self):
        return self.__is_read

    def mark_read(self):
        self.__is_read = True

    def __str__(self):
        state = "read" if self.__is_read else "unread"
        return f"Notification #{self.notification_id} [{self.kind}] {self.message} ({state})"


In [ ]:
# Quick demo: run this cell to check all the classes work together
inv = Inventory()
milk = make_product(1, "Amul Milk 1L", 30, "Dairy")
chips = make_product(2, "Lays Classic", 20, "Snacks")
inv.restock(1, 50)
inv.restock(2, 100)

cust = Customer(101, "Billa", "9876543210", "billa@example.com", "Delhi", "2000-01-01", "M", "110001")
rider = Rider(201, "Ramesh", "9123456780", "ramesh@example.com", "Delhi")

print(cust.cart.add(milk, 2))
print(cust.cart.add(chips, 3))
order = cust.place_order(inv)
print(order)

pay = Payment(order, "upi")
print(pay.pay())

d = Delivery(order, rider)
print(d.advance())
print(d.advance())
print(d)

print(Coupon("SAVE10", 10, 50, min_order=50).apply(order.total()))
print(Review(order, 5, "Fast delivery"))

wl = Wishlist(cust)
wl.add(milk)
print(wl.move_to_cart(milk))
print(Notification(cust, "Order delivered", "delivery"))
